# Validation: Parrott's helicopter exhaust muffler (NASA TN D-7309)

Reference: T. L. Parrott, *An improved method for design of expansion-chamber mufflers with application to an operational helicopter*, NASA TN D-7309 (1973).

This notebook validates the Nefes acoustic-network layer against a landmark reactive-muffler study.
Parrott's report gave one of the first muffler-design methods to account for the mean exhaust flow, and its three-stage muffler is the device reproduced (as a dual-tailpipe adaptation) in E. Dokumacı, *Duct Acoustics* (Cambridge, 2021), Fig 5.27.
Unlike a textbook expansion chamber, this is a real, flight-tested silencer: three concentric extended-tube expansion chambers in series, with fully published geometry, operating state, a computed transmission-loss spectrum, and measured field data.

**Why this case.**
The geometry is tabulated in full (Fig 13), so nothing has to be digitised to build the model.
It exercises the network *as a network*: twelve components, three chambers, and six closed annular side branches.
It carries a mean flow (tailpipe Mach $0.1$), which Nefes solves for rather than assumes.
And it comes with two independent references: the classical transfer-matrix method, and measured transmission losses (Table II / Fig 17).

**Operating state (Table I).**
The exhaust runs at $\overline{T} = 1090\ \mathrm{K}$ with a design sound speed $\overline{c} = 610\ \mathrm{m/s}$ (2000 ft/s) and a tailpipe Mach number $\overline{M} = 0.1$.
Transmission loss depends on the gas only through $\overline{c}$, so we model the medium as a calorically perfect gas tuned to that sound speed.

**Plan.**
1. **Part 1** — a single expansion chamber against the exact analytic transmission loss (the machine-precision anchor).
2. **Part 2** — the full three-stage muffler: the Nefes network against the classical transfer-matrix method, with the measured points overlaid.
3. **Part 2b** — Parrott's own published curve (digitised), and the branch-wall damping that carries the lossless network onto it.
4. **Part 3** — the mean flow Nefes solves for, and what it does (and does not) do to the transmission loss.


In [ ]:
import numpy as np
import plotly.graph_objects as go

import nefes
from nefes.elements import catalog as cat
from nefes.perturbation import PerturbationBC
from nefes.plotting import COLORWAY, use_nefes_theme

use_nefes_theme()

# --- exhaust gas: calorically perfect, tuned to the reported sound speed --------
# Parrott's exhaust runs at T = 1090 K (Table I); the design sound speed is
# c = 610 m/s (2000 ft/s).  Transmission loss depends on the gas only through c,
# so we model the medium as a perfect gas whose sound speed equals 610 m/s.
GAMMA, R = 1.4, 287.0
GAS = nefes.perfect_gas(R, GAMMA)
CO = 610.0  # exhaust sound speed [m/s]  (2000 ft/s)
P0 = 101325.0  # tailpipe static pressure [Pa]
T0 = CO**2 / (GAMMA * R)  # temperature giving c == CO
RHO0 = P0 / (R * T0)

# --- muffler geometry (Fig 13) --------------------------------------------------
S1 = 0.002  # pipe / tailpipe cross-sectional area [m^2]
SC = 0.019  # expansion-chamber cross-sectional area [m^2]
SANN = SC - S1  # annular dead-space area around an extended tube
M_RATIO = SC / S1  # chamber/pipe area ratio

# engine-bank firing frequency: rotor 356 rpm / mast gear ratio 0.111 -> 3207 rpm;
# f1 = Nc * rpm / 120 with Nc = 3  ->  ~80.2 Hz
F1 = 3.0 * (356.0 / 0.111) / 120.0
FREQS = np.linspace(20.0, 1500.0, 1500)

# measured muffler insertion loss ~ transmission loss (Table II / Fig 17)
HARM_N = np.array([1, 2, 3, 4, 5, 6])
HARM_F = HARM_N * F1
HARM_TL = np.array([11.5, 24.5, 8.5, 9.8, -14.0, -9.5])

print(f"c = {CO} m/s   T0 = {T0:.1f} K   rho0 = {RHO0:.4f} kg/m^3")
print(f"chamber/pipe area ratio m = {M_RATIO:.2f}")
print(f"engine-bank firing fundamental f1 = {F1:.1f} Hz")


def tl_two_port(sol, freqs, edge_in, edge_out, freeze=None):
    """Transmission loss of an unbranched 2-port from the acoustic scattering matrix.

    Reads the transmission coefficient tau between the (equal-area) inlet and outlet
    pipe edges; TL = -20 log10|tau|.  ``freeze`` folds the listed wall terminals into
    the operator at their declared closure (rigid by default, or an absorptive
    reflection in Part 2b), so the closed stubs reduce to a clean inlet->outlet two-port.
    """
    resp = sol.perturbation_response(freqs, freeze=freeze) if freeze else sol.perturbation_response(freqs)
    S = resp.acoustic_scattering_matrix(edge_in, edge_out)
    tau = S[:, 1, 0]  # outgoing@outlet per incoming@inlet
    return -20.0 * np.log10(np.abs(tau))

## Part 1 — a single expansion chamber (the exact anchor)

Parrott's Fig 1 is the simplest reactive silencer: an inlet pipe of area $S_1$, a sudden expansion to the chamber area $S_c$, a length $L$, a contraction back to $S_1$, and the outlet pipe.
With reflection-free (anechoic) ends and no mean flow, its transmission loss has a closed form,

$$\mathrm{TL} = 10\log_{10}\!\Big[1 + \tfrac14\big(m - \tfrac1m\big)^2\sin^2(kL)\Big],\qquad k = \frac{2\pi f}{\overline{c}},\quad m = \frac{S_c}{S_1},$$

which is the no-flow limit of Parrott's Eq (9).
We build the literal geometry as a `Network`, read the transmission coefficient from the acoustic scattering matrix, and compare.


In [ ]:
def build_plain_chamber(mach, length):
    """Parrott's Fig-1 expansion chamber: pipe -> expansion -> chamber -> contraction -> pipe."""
    mdot = RHO0 * (mach * CO) * S1
    net = nefes.Network(GAS)
    inlet = net.add(cat.mass_flow_inlet(mdot, T0))
    expansion = net.add(cat.isentropic_area_change("expansion"))
    chamber = net.add(cat.duct(length))
    contraction = net.add(cat.isentropic_area_change("contraction"))
    outlet = net.add(cat.pressure_outlet(P0, T0))
    e_in = net.connect(inlet, expansion, S1)
    net.connect(expansion, chamber, SC)
    net.connect(chamber, contraction, SC)
    e_out = net.connect(contraction, outlet, S1)
    sol = net.solve()
    assert sol.converged, sol.print_residuals()
    return sol, e_in, e_out


def tl_chamber_analytic(freqs, length):
    """Textbook expansion-chamber TL (Parrott Eq 9 limit, no flow, anechoic ends)."""
    k = 2.0 * np.pi * freqs / CO
    return 10.0 * np.log10(1.0 + 0.25 * (M_RATIO - 1.0 / M_RATIO) ** 2 * np.sin(k * length) ** 2)


L_DEMO = 0.61
sol_p, ep_in, ep_out = build_plain_chamber(0.0, L_DEMO)
tl_p = tl_two_port(sol_p, FREQS, ep_in, ep_out)
tl_p_ref = tl_chamber_analytic(FREQS, L_DEMO)
err_p = np.max(np.abs(tl_p - tl_p_ref))
print(f"[Part 1] max|Nefes - analytic| = {err_p:.2e} dB   TLmax = {tl_p.max():.2f} dB")
assert err_p < 1e-9, err_p

In [ ]:
fig1 = go.Figure()
fig1.add_trace(
    go.Scatter(x=FREQS, y=tl_p_ref, name="analytic (Parrott Eq 9)", line=dict(color="black", width=5), opacity=0.35)
)
fig1.add_trace(go.Scatter(x=FREQS, y=tl_p, name="Nefes network", line=dict(color=COLORWAY[0], width=1.8)))
fig1.update_layout(
    title="Part 1 - single expansion chamber: Nefes vs exact analytic",
    xaxis_title="Frequency, Hz",
    yaxis_title="Transmission loss, dB",
    width=820,
    height=420,
    legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"),
)
fig1

The Nefes network reproduces the analytic curve to machine precision: the lobes peak at $\mathrm{TL}_{\max} = 10\log_{10}[1+\tfrac14(m-1/m)^2] \approx 13.6\ \mathrm{dB}$ for this area ratio, and the nulls fall at $f = n\,\overline{c}/2L$.
This fixes the transmission-loss machinery before we assemble the real muffler.


## Part 2 — the three-stage helicopter muffler

The optimised muffler (Fig 13) is a series of three concentric extended-tube expansion chambers.
Its twelve components, working from the tailpipe back to the inlet, are:

| # | Component | Length [m] | Area [m$^2$] |
|---:|---|---:|---:|
| 1 | Tailpipe | 0.60 | 0.002 |
| 2 | Extended outlet | 0.06 | 0.002 |
| 3 | First chamber | 0.25 | 0.019 |
| 4 | Extended inlet | 0.04 | 0.002 |
| 5 | Connector | 0.01 | 0.002 |
| 6 | Extended outlet | 0.08 | 0.002 |
| 7 | Second chamber | 0.51 | 0.019 |
| 8 | Extended inlet | 0.30 | 0.002 |
| 9 | Connector | 0.61 | 0.002 |
| 10 | Extended outlet | 0.13 | 0.002 |
| 11 | Third chamber | 0.76 | 0.019 |
| 12 | Extended inlet | 0.44 | 0.002 |

**How the geometry maps to elements.**
Each chamber is entered by an inlet tube that reaches a distance $L_\mathrm{in}$ into it, and is left by an outlet tube that reaches $L_\mathrm{out}$ in from the other end.
Acoustically, the tube tip is an area change ($S_1 \leftrightarrow S_c$) carrying the through flow, and the annulus around the tube — closed by the chamber end wall — is a rigid side branch of length equal to the tube's reach and area $S_c - S_1$.
So each chamber becomes: an inlet-tube tip (expansion, plus a closed inlet stub), the open chamber body of length $L_c - L_\mathrm{in} - L_\mathrm{out}$, and an outlet-tube entrance (contraction, plus a closed outlet stub).
The pipe joining two chambers is the extended outlet of the upstream chamber, the external connector, and the extended inlet of the downstream chamber in series.
The stubs are the tuning elements; keeping them rigid during the measurement folds them into the operator and leaves a clean inlet$\to$outlet two-port.


In [ ]:
CHAMBERS = [  # (chamber length, extended-inlet length, extended-outlet length) [m]
    (0.76, 0.44, 0.13),  # third chamber (nearest the exhaust inlet)
    (0.51, 0.30, 0.08),  # second chamber
    (0.25, 0.04, 0.06),  # first chamber (nearest the tailpipe)
]
CONNECTORS = [1.04, 0.13]  # S1 pipe between chambers = ext_out + connector + ext_in
TAILPIPE = 0.66  # ext_out(first chamber) + tailpipe


def add_extended_chamber(net, node_in, Lch, Lin, Lout, walls, wall_reflection=None):
    """One concentric extended-tube chamber: an inlet-tube tip (expansion + inlet
    annular stub), the open chamber body, and an outlet-tube entrance (contraction +
    outlet annular stub).  Returns the outgoing S1 node.

    ``wall_reflection`` sets the stub end-wall closure: ``None`` is a rigid wall (R=1);
    a value ``R < 1`` gives an absorptive wall used in Part 2b.
    """

    def new_wall():
        if wall_reflection is None:
            return net.add(cat.wall())
        return net.add(cat.wall(perturbation_bc=PerturbationBC.reflection(wall_reflection)))

    Lmid = Lch - Lin - Lout
    tip_in = net.add(cat.junction())  # inlet-tube tip
    body = net.add(cat.duct(Lmid))  # open chamber region (SC)
    tip_out = net.add(cat.junction())  # outlet-tube entrance
    stub_in = net.add(cat.duct(Lin))
    wall_in = new_wall()
    stub_out = net.add(cat.duct(Lout))
    wall_out = new_wall()
    net.connect(node_in, tip_in, S1)  # incoming inlet tube (S1)
    net.connect(tip_in, body, SC)  # expansion into chamber body
    net.connect(tip_in, stub_in, SANN)  # inlet annular stub
    net.connect(stub_in, wall_in, SANN)
    net.connect(body, tip_out, SC)  # chamber body -> outlet-tube entrance
    net.connect(tip_out, stub_out, SANN)  # outlet annular stub
    net.connect(stub_out, wall_out, SANN)
    walls.extend([wall_in, wall_out])
    return tip_out


def build_muffler(mach, wall_reflection=None):
    """Parrott's three-stage muffler (Fig 13) as a Nefes network; returns
    (net, sol, inlet_edge, outlet_edge, stub_walls).

    ``wall_reflection`` closes the stub end walls at a reflection R < 1 (absorptive,
    Part 2b); the default rigid walls are the lossless case.
    """
    mdot = RHO0 * (mach * CO) * S1
    net = nefes.Network(GAS)
    walls = []
    inlet = net.add(cat.mass_flow_inlet(mdot, T0))
    inlet_tube = net.add(cat.duct(CHAMBERS[0][1]))  # extended inlet of chamber 3
    e_in = net.connect(inlet, inlet_tube, S1)
    node = inlet_tube
    for i, (Lch, Lin, Lout) in enumerate(CHAMBERS):
        node = add_extended_chamber(net, node, Lch, Lin, Lout, walls, wall_reflection)
        if i < len(CONNECTORS):
            link = net.add(cat.duct(CONNECTORS[i]))
            net.connect(node, link, S1)
            node = link
    tail = net.add(cat.duct(TAILPIPE))
    net.connect(node, tail, S1)
    outlet = net.add(cat.pressure_outlet(P0, T0))
    e_out = net.connect(tail, outlet, S1)
    sol = net.solve()
    return net, sol, e_in, e_out, walls


net_m, sol_m, em_in, em_out, walls_m = build_muffler(0.0)
assert sol_m.converged, sol_m.print_residuals()
sol_m.verify()
Mabs = np.abs(sol_m.field("M"))
print(f"[Part 2] muffler converged; residual {sol_m.residual_norm:.1e}; max Mach {Mabs.max():.3f}")

The network Nefes assembled from that geometry:

In [ ]:
net_m.plot()

In [ ]:
def tl_transfer_matrix(freqs, r_in=1.0, r_out=1.0):
    """Classical 1-D four-pole (p, U=S*v) transmission loss of the same muffler.

    Area changes are transparent in the (pressure, volume-velocity) variables; a
    stub of area S and length l terminated by an end reflection R is the shunt
    impedance Zb = (rho c/S)(1+G)/(1-G) with G = R exp(-2 i k l).  Rigid stubs
    (R=1) give the lossless -i(rho c/S)cot(kl); r_in/r_out < 1 add the branch-wall
    damping Parrott used.  This is the same reactive-muffler method Parrott used, in
    its no-flow limit, and serves as an independent reference for the Nefes network.
    """
    Z1, Z2, Zann = RHO0 * CO / S1, RHO0 * CO / SC, RHO0 * CO / SANN

    def duct(k, Z, L):
        kl = k * L
        return np.array([[np.cos(kl), 1j * Z * np.sin(kl)], [1j * np.sin(kl) / Z, np.cos(kl)]])

    def stub(k, L, Rend):
        G = Rend * np.exp(-2j * k * L)
        Zb = Zann * (1.0 + G) / (1.0 - G)
        return np.array([[1.0, 0.0], [1.0 / Zb, 1.0]])

    out = np.empty_like(freqs)
    for j, f in enumerate(freqs):
        k = 2.0 * np.pi * f / CO
        chain = [duct(k, Z1, CHAMBERS[0][1])]
        for i, (Lch, Lin, Lout) in enumerate(CHAMBERS):
            chain += [stub(k, Lin, r_in), duct(k, Z2, Lch - Lin - Lout), stub(k, Lout, r_out)]
            chain.append(duct(k, Z1, CONNECTORS[i] if i < len(CONNECTORS) else TAILPIPE))
        T = np.eye(2, dtype=complex)
        for Mx in chain:
            T = T @ Mx
        out[j] = 20.0 * np.log10(0.5 * abs(T[0, 0] + T[0, 1] / Z1 + Z1 * T[1, 0] + T[1, 1]))
    return out


tl_m = tl_two_port(sol_m, FREQS, em_in, em_out, freeze=walls_m)
tl_m_ref = tl_transfer_matrix(FREQS)
d_m = np.abs(tl_m - tl_m_ref)
print(
    f"[Part 2] Nefes vs transfer-matrix over {FREQS[0]:.0f}-{FREQS[-1]:.0f} Hz: "
    f"rms {np.sqrt(np.mean(d_m**2)):.1e} dB, max {d_m.max():.1e} dB "
    f"(max at a >100 dB stub resonance where cot(kl) is singular)"
)
assert d_m.max() < 1e-2, d_m.max()
tl_at_harm = np.interp(HARM_F, FREQS, tl_m)
for n, f, meas, comp in zip(HARM_N, HARM_F, HARM_TL, tl_at_harm):
    print(f"    H{n}  f={f:6.1f} Hz   Nefes TL={comp:5.1f} dB   measured={meas:+5.1f} dB")

In [ ]:
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(x=FREQS, y=tl_m_ref, name="transfer-matrix reference", line=dict(color="black", width=5), opacity=0.30)
)
fig2.add_trace(go.Scatter(x=FREQS, y=tl_m, name="Nefes network", line=dict(color=COLORWAY[1], width=1.4)))
fig2.add_trace(
    go.Scatter(
        x=HARM_F[:4],
        y=HARM_TL[:4],
        name="measured (Fig 17)",
        mode="markers",
        marker=dict(color=COLORWAY[3], size=11, symbol="circle-open", line=dict(width=3)),
    )
)
fig2.update_layout(
    title="Part 2 - three-stage helicopter muffler: Nefes vs transfer-matrix + measured",
    xaxis_title="Frequency, Hz",
    yaxis_title="Transmission loss, dB",
    width=900,
    height=470,
    legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"),
)
fig2.update_yaxes(range=[0, 80])
fig2

**Verification.**
Nefes builds this muffler from physical geometry — areas read off the edges, mass and energy conserved at every junction — and reproduces the classical transfer-matrix (four-pole) method to machine precision across the whole band.
The only non-negligible residual sits on a razor-sharp stub resonance above $100\ \mathrm{dB}$, where the reference's $\cot(kl)$ is singular; the root-mean-square difference is at the $10^{-5}\ \mathrm{dB}$ level.
The two computations share no code: the reference multiplies $2\times2$ four-pole matrices in the $(p, U)$ variables, while Nefes solves the linearised perturbation network and extracts the scattering matrix.

**Measured comparison.**
The open circles are the muffler's measured transmission losses at the first four engine-bank firing harmonics (Table II / Fig 17), obtained from insertion-loss measurements on the flying aircraft.
As in Parrott's own Fig 17, the reactive-muffler prediction captures where the attenuation lives but sits *above* the measured points: the field estimates are lower bounds, contaminated by the finite source impedance and by muffler self-noise.
The verification claim is the machine-precision agreement with the transfer-matrix reference; the measured points are the real-world context, taken up next.


## Part 2b — the published curve, and the role of branch damping

Part 2 compared the Nefes network against the *idealised* transfer-matrix method, in which the stub end walls are perfectly rigid.
That model has razor-sharp resonances (above $100\ \mathrm{dB}$) and nulls that reach zero, because a lossless side branch stores and returns all of its energy.
Parrott's own published curve (Fig 9) is smoother: its peaks sit near $45\ \mathrm{dB}$ and its nulls near $20$–$30\ \mathrm{dB}$.

The difference is dissipation.
Real end caps are not perfectly rigid; they yield and absorb a little, and Parrott represented this by giving the branch end walls a reflection factor below unity (he used $0.8$ on the contraction branches).
In Nefes the same physics is a boundary condition on the stub walls: closing each wall at a reflection $R<1$ instead of leaving it rigid.
Below we give all six stub walls $R = 0.8$ (Parrott applied it to the contraction branches only, and handled the expansion loss inside his junction model), then **freeze** them.
Freezing keeps a terminal at its *declared* boundary condition and folds it into the operator, so an absorptive closed stub reduces to the same clean inlet$\to$outlet two-port as a rigid one: the `tl_two_port` reader is unchanged, and nothing has to be condensed by hand.
The result matches an independent lossy transfer-matrix to machine precision, so the loss it adds is physical, not numerical.

The comparison overlays the lossless and absorptive Nefes curves with Parrott's Fig 9 (digitised for this muffler) and the measured points.


In [ ]:
# Parrott's Fig 9 "Calculated" curve, digitized (log10(lambda_ft), TL[dB]); lambda in feet.
PARROTT_FIG9 = np.array(
    [
        (0.18, 37.0),
        (0.20, 39.0),
        (0.22, 31.0),
        (0.25, 44.0),
        (0.28, 45.0),
        (0.31, 45.0),
        (0.33, 41.0),
        (0.36, 22.0),
        (0.37, 20.0),
        (0.40, 33.0),
        (0.43, 43.0),
        (0.45, 44.0),
        (0.48, 39.0),
        (0.51, 31.0),
        (0.53, 30.0),
        (0.55, 34.0),
        (0.57, 33.0),
        (0.60, 45.0),
        (0.63, 41.0),
        (0.66, 31.0),
        (0.69, 33.0),
        (0.72, 40.0),
        (0.76, 44.0),
        (0.79, 38.0),
        (0.82, 28.0),
        (0.84, 26.0),
        (0.88, 29.0),
        (0.92, 28.0),
        (0.96, 29.0),
        (1.00, 30.0),
        (1.03, 31.0),
        (1.08, 31.0),
        (1.12, 30.0),
        (1.18, 27.0),
        (1.24, 22.0),
        (1.30, 17.0),
        (1.36, 12.0),
        (1.42, 8.5),
    ]
)
C_FTS = CO / 0.3048  # sound speed in ft/s (Parrott's wavelength axis is in feet)
PARROTT_F = C_FTS / 10.0 ** PARROTT_FIG9[:, 0]  # frequency [Hz]
PARROTT_TL = PARROTT_FIG9[:, 1]

# Absorptive stub walls: give each end wall a reflection R < 1, then freeze it.  A frozen
# terminal folds in at its *declared* boundary condition (rigid, or here R = 0.8), so the
# closed absorptive stubs reduce to the same clean inlet->outlet two-port as the rigid case:
# the tl_two_port reader is unchanged, no manual condensation needed.
R_WALL = 0.8  # absorptive end-wall reflection (Parrott's branch value)
net_d, sol_d, ed_in, ed_out, walls_d = build_muffler(0.0, wall_reflection=R_WALL)
tl_damped = tl_two_port(sol_d, FREQS, ed_in, ed_out, freeze=walls_d)
# independent check: the absorptive-wall Nefes result equals the lossy transfer-matrix
tl_damped_ref = tl_transfer_matrix(FREQS, r_in=R_WALL, r_out=R_WALL)
err_d = np.max(np.abs(tl_damped - tl_damped_ref))
print(f"[Part 2b] Nefes absorptive-stub vs lossy transfer-matrix: max {err_d:.2e} dB")
assert err_d < 1e-5, err_d
print(
    f"[Part 2b] peak TL: lossless {tl_m.max():.0f} dB -> damped {tl_damped.max():.0f} dB; "
    f"deep nulls fill from {tl_m.min():.0f} to {tl_damped.min():.0f} dB"
)

In [ ]:
fig2b = go.Figure()
fig2b.add_trace(
    go.Scatter(x=FREQS, y=tl_m, name="Nefes, lossless (rigid stubs)", line=dict(color=COLORWAY[1], width=1.0))
)
fig2b.add_trace(
    go.Scatter(
        x=FREQS, y=tl_damped, name=f"Nefes, absorptive stubs (R={R_WALL})", line=dict(color=COLORWAY[4], width=2.2)
    )
)
fig2b.add_trace(
    go.Scatter(
        x=PARROTT_F,
        y=PARROTT_TL,
        name="Parrott Fig 9 (digitized)",
        line=dict(color="black", width=2.4, dash="dash"),
        mode="lines+markers",
        marker=dict(size=4),
    )
)
fig2b.add_trace(
    go.Scatter(
        x=HARM_F[:4],
        y=HARM_TL[:4],
        name="measured (Fig 17)",
        mode="markers",
        marker=dict(color=COLORWAY[3], size=11, symbol="circle-open", line=dict(width=3)),
    )
)
fig2b.update_layout(
    title="Part 2b - branch-wall damping carries the lossless network toward Parrott's curve",
    xaxis_title="Frequency, Hz",
    yaxis_title="Transmission loss, dB",
    width=900,
    height=470,
    legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"),
)
fig2b.update_xaxes(range=[20, 1300])
fig2b.update_yaxes(range=[0, 70])
fig2b

**What the comparison shows.**
The resonance *structure* is independent of the damping and of which method computes it: the peak and null frequencies of the lossless Nefes curve, the absorptive Nefes curve, and Parrott's published curve coincide.
That aligns the geometry reading against Parrott's own computation, not only against a re-implementation of the same method.
Adding the absorptive end walls then carries the lossless network down onto Parrott's curve and the measured points — the sharp peaks round toward $45\ \mathrm{dB}$ and the deep nulls fill to $10$–$30\ \mathrm{dB}$.
A residual gap remains at the peaks because Parrott's full model also carries the Mach-shifted tailpipe radiation impedance and a non-isentropic expansion loss that this purely reactive, anechoic model does not; the narrower point is made cleanly here — the branch-wall absorption is what turns the idealised lossless response into the measured one.


## Part 3 — the mean flow Nefes solves for

Parrott, like every transfer-matrix treatment, *assumes* a uniform tailpipe Mach number.
Nefes instead solves the compressible mean flow first, then linearises the acoustics on top of it.
Rebuilding the muffler with the design mass flow ($\overline{M} = 0.1$ in the tailpipe) lets us read the actual Mach distribution and ask what the flow does to the transmission loss.


In [ ]:
net_f, sol_f, ef_in, ef_out, walls_f = build_muffler(0.1)
assert sol_f.converged, sol_f.print_residuals()
sol_f.verify()
Mf = np.abs(sol_f.field("M"))
Mpipe = float(Mf[ef_out])  # tailpipe (S1)
Mchamber = Mpipe * S1 / SC  # continuity: chamber body decelerates by the area ratio
print(
    f"[Part 3] mean-flow solve: tailpipe/pipes M={Mpipe:.3f}, chamber bodies M~{Mchamber:.4f}, "
    f"stubs M=0 (closed); max M={Mf.max():.3f}"
)

tl_f = tl_two_port(sol_f, FREQS, ef_in, ef_out, freeze=walls_f)
smooth = FREQS <= 200.0  # smooth low-frequency band, away from razor-sharp stub resonances
print(
    f"[Part 3] anechoic TL is nearly flow-independent: max|TL(M=0.1)-TL(M=0)| = "
    f"{np.max(np.abs(tl_f - tl_m)[smooth]):.2f} dB below 200 Hz; sharp resonances shift ~0.1%."
)

In [ ]:
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=FREQS, y=tl_m, name="quiescent (M=0)", line=dict(color=COLORWAY[0], width=2.4)))
fig3.add_trace(
    go.Scatter(x=FREQS, y=tl_f, name="mean flow (M=0.1)", line=dict(color=COLORWAY[4], width=1.3, dash="dot"))
)
fig3.update_layout(
    title="Part 3 - the anechoic transmission loss is nearly flow-independent (M=0 vs M=0.1)",
    xaxis_title="Frequency, Hz",
    yaxis_title="Transmission loss, dB",
    width=900,
    height=420,
    legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"),
)
fig3.update_yaxes(range=[0, 80])
fig3

**What the mean flow does.**
Nefes recovers the operating point Parrott assumed — Mach $0.1$ in the pipes, an order of magnitude less in the wide chamber bodies, and identically zero in the closed stubs — and confirms it is self-consistent.
Its effect on the *anechoic* transmission loss is small: below $0.3\ \mathrm{dB}$ across the smooth low-frequency band, with the sharp resonances shifting by about $0.1\%$ (the convective correction $kL \to kL/(1-\overline{M}^2)$, felt only on the flow-carrying pipes).
This is exactly Parrott's own finding that the internal flow corrections are "relatively insignificant".

**A note on Parrott's Fig 6.**
Parrott's dramatic "with-flow / without-flow" difference is *not* a contradiction of the near-invariance seen here.
That figure is computed with the tailpipe's radiation impedance as the termination, so its "transmission loss" is termination-dependent and the flow enters mainly through the Mach-shifted radiation reflection factor — an insertion-loss / radiated-power effect.
The transmission loss used throughout this notebook is the modern, termination-independent (anechoic) quantity, which is why it barely moves with flow.
Nefes represents the radiating case separately, through the outlet boundary condition (`PerturbationBC.mean_flow_open_end()`), rather than through the scattering-matrix transmission loss.
